# Preliminary test of a simple Lipschitz Network

The preliminary tests here is structured as follows:
- Data exploration and preprocessing. Includes a simple estimation of the Lipschitz constant of the data.
- Layer design
  - Activation function: GroupSort
  - Linear layer: Spectral Normalization (likely)
- Architecture design
- Training and Evaluation
- Error analysi and dicsussion of results

## Data exploration and preprocessing

The data is provided by Joaquin Iturizza for this project and consists of...

## Layer design

From the literature review we have seen that the state of the art Lipschitz networks are based on GroupSort activation functions. Further, the LHCb collaboration are using spectral normalization in their networks. The method of spectral normalization could likely be improved upon by state of the art layer designs, but little effort is made into regression problems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

GroupSort is an activation function that is universally expressive for 1-Lipschitz functions. It preserves the gradient norm, a key difference from the ReLU activation function. GroupSort separates the input feature vectors into groups of size $n_\text{groups}$, and sorts the values in each group in ascending order. The permutation group is an preserves the p-norm, thus keeping the gradient norm as well. The non-linearity is introduced by the permutation operation.


In [ ]:
# Code from : https://github.com/niklasnolte/MonotonicNetworks/blob/main/monotonicnetworks/group.py

class GroupSort(nn.Module):
    def __init__(self, num_groups:int, axis: int = -1):
        super(GroupSort, self).__init__()
        self.num_groups = num_groups
        self.axis = axis
    def forward(self, x: torch.Tensor):
        return group_sort(x, self.n_groups, self.axis)

    def extra_repr(self):
        return f"num_groups: {self.n_groups}"
        pass

def get_sorting_shape(x: torch.Tensor, n_groups: int, axis: int) -> list:
    shape = list(x.shape)
    num_features = shape[axis]
    if num_features % n_groups:
        raise ValueError(
            f"number of features({num_features}) needs to be a multiple of n_groups({n_groups})"
        )
    shape[axis] = -1
    n_per_group = num_features // n_groups
    if axis == -1:
        shape.append(n_per_group)
    else:
        shape.insert(axis + 1, n_per_group)
    return shape



def group_sort(x: torch.Tensor, n_groups: int, axis: int = -1) -> torch.Tensor:
    if x.shape[0] == 0:
        return x
    size = get_sorting_shape(x, n_groups, axis)
    grouped_x = x.view(*size) # Changes the shape of the input tensor to a group along the specified axis
    sort_dim = axis if axis == -1 else axis +1
    sorted_grouped_x, _ = grouped_x.sort(dim=sort_dim)
    sorted_x = sorted_grouped_x.view(*x.shape) # Reshapes the sorted tensor back to the original shape
    return sorted_x



In [ ]:
# TESTING THE GROUPSORT ACTIVATION FUNCTION
# Creating dummy data for testing
x = torch.tensor([[3.0, 1.0, 2.0, 6.0, 8.0, 3.0, 9.0, 11.], [5.0, 4.0, 8.0, 7.0, 11.0, 2.0, 1.0, 13.]])
n_groups = 2 # Using 1 group turns this into a FullSort activation function

# Applying the GroupSort activation function
sorted_x = group_sort(x, n_groups)

# Printing the original and sorted tensors
print("Original Tensor:")
print(x)
print("\nSorted Tensor:")
print(sorted_x)

Original Tensor:
tensor([[ 3.,  1.,  2.,  6.,  8.,  3.,  9., 11.],
        [ 5.,  4.,  8.,  7., 11.,  2.,  1., 13.]])

Sorted Tensor:
tensor([[ 1.,  2.,  3.,  6.,  3.,  8.,  9., 11.],
        [ 4.,  5.,  7.,  8.,  1.,  2., 11., 13.]])


Following the procedure in "Robust and Provably Monotonic Networks" by Kitouni et al. (2023) from the work they made on the LHCb collaboration, we will use the weight normalization in order to Lipschitz constrain a small network. In this case the network is a simple feed forward network with linear layers and GroupSort activations. Note, the weight normalization here works only with the $l_1$-norm (the Manhatten or Taxi-cab norm).

The can be seen as a nested function of applications of weight matrices $W^i$ and activation functions $\sigma^i$, where the superscript $i$ denotes the layer number. A constrained network must satisfy the following condition for all layers $i$:
$$
\sum^i_{i=0}||W^i||_1 \leq \lambda
$$

The weight normalization is defined by the rescaling of the wright matrix $W$ as 
$$
W^i \rightarrow W^{'i} = \frac{W^i}{\max{}(1, \lambda^{-1/m}||W^i||_1)}. 
$$

For the purpose of constraining the weight matrices, according to $\lambda$ it suffices to rescale the columns of each weight matrix $W^i$.
$$
W^i \rightarrow W^{'i} = W^i \text{diag} \bigg( \frac{1}{\max{}(1, \lambda^{-1/m}||W^i||_1)}\bigg) .
$$


In [ ]:
# Linear layer with weight normalization
# Source: https://github.com/niklasnolte/MonotonicNetworks/blob/main/monotonicnetworks/LipschitzMonotonicNetwork.py

kind = {} # Dictionary of norm types for the weight normalization

class LipschitzLinear(nn.Linear): # Inherits from the Linear layer in torch
    def __init__(self, in_features: int, out_features: int, bias: bool = True, Lipschitz_const: float = 1.0, kind: str = "one"):
        super(LipschitzLinear, self).__init__(in_features, out_features, bias=bias)
        self.Lipschitz_const = Lipschitz_const
        self.kind = kind
        self.weight = nn.Parameter(self.weight.data)
        self.reset_parameters()

        self.register_buffer("weight_norm", torch.Tensor([Lipschitz_const]))


In [ ]:
# Utregning av vinkveld
vin_liquiderie = 37.0
vin_deux_amies = 42.0 
ol_deux_amies = 3.6
mat_deux_amies = 13.0 + 8.50

# Oppdeling
vin_pr_pers = (vin_liquiderie + vin_deux_amies) / 3

mat_pr_pers = (mat_deux_amies) / 2

# Fordeling av kostnader
emil = vin_pr_pers + mat_pr_pers
ella = vin_pr_pers + mat_pr_pers + ol_deux_amies
jonas = vin_pr_pers

print(f"Emil: {emil:.2f} EUR")
print(f"Ella: {ella:.2f} EUR")
print(f"Jonas: {jonas:.2f} EUR")

# Balansesjekk
total_kostnad = vin_liquiderie + vin_deux_amies + ol_deux_amies + mat_deux_amies
total_fordelt_kostnad = emil + ella + jonas
print(f"Total kostnad: {total_kostnad:.2f} EUR")
print(f"Total fordelt kostnad: {total_fordelt_kostnad:.2f} EUR")



Emil: 37.08 EUR
Ella: 40.68 EUR
Jonas: 26.33 EUR
Total kostnad: 104.10 EUR
Total fordelt kostnad: 104.10 EUR
